In [1]:
using Glob 
using DelimitedFiles, DataFrames, EzXML
using Dates
using LinearAlgebra
using ForwardDiff
using Statistics
using Interpolations
using Rotations
using Unitful
using ThreadsX
using WENO4
using Polyester
using CoordinateTransformations
import PhysicalConstants.CODATA2018: m_p, m_e, e, ε_0, μ_0, c_0, G, k_B
const M_Mars = 0.64169e24u"kg"
const R_Mars = 3389.5u"km"  #m
const au = 149597870.7u"km"

include("../../src/scripts/MAVEN_load.jl")
include("../../src/scripts/MAVEN_STATIC.jl")
include("../../src/scripts/SphericalSplines.jl")
using .MAVEN_STATIC
using .MAVEN_load
using .SphericalSplines

date = Date(2022, 8, 24)
datestr = Dates.format.(date, "yyyymmdd")
year_num = Dates.format(date, "yyyy")
month_num = Dates.format(date, "mm")

ts = DateTime(date) + Dates.Hour(7) + Dates.Minute(26)
te = ts + Dates.Minute(3)

data_dir = "E:\\Data"
output_dir = "E:\\Output"
MAVEN_path = joinpath(data_dir, "MAVEN")
MAG_path = joinpath(MAVEN_path, "MAG", "l2", year_num, month_num)
STATIC_path = joinpath(MAVEN_path, "STATIC", "l2", year_num, month_num)
MAVEN_output_path = joinpath(output_dir, "MAVEN")
STATIC_output_path = joinpath(MAVEN_output_path, "STATIC")
VDF_path = joinpath(STATIC_output_path, "VDF", year_num, month_num)
mkpath(VDF_path)
MAG_file = glob("mvn_mag_l2*$datestr*.sts", MAG_path)
STA_file = glob("mvn_sta_l2*d1*32e4d16a8m*$datestr*.cdf", STATIC_path)
println("MAG file: ", MAG_file)
println("STATIC file: ", STA_file)
println("Output path: ", VDF_path)

mag_data = MAVEN_load.load_mag_l2(MAG_file[1])
sta_data = MAVEN_load.load_cdf(STA_file[1])
println("Data loaded.")

MAG file: ["E:\\Data\\MAVEN\\MAG\\l2\\2022\\08\\mvn_mag_l2_2022236ss_20220824_v01_r01.sts"]
STATIC file: ["E:\\Data\\MAVEN\\STATIC\\l2\\2022\\08\\mvn_sta_l2_d1-32e4d16a8m_20220824_v02_r03.cdf"]
Output path: E:\Output\MAVEN\STATIC\VDF\2022\08
Data loaded.


In [2]:
# MVN MAG
mag_ut = mag_data[:JulUTtime] .- datetime2julian(DateTime(date))
mag_B_mso = mag_data[:B]
mag_ind = ThreadsX.findall(ts .<= mag_data[:epoch] .<= te)
mag_B_mean_mso = dropdims(mean(Matrix(mag_B_mso[mag_ind, :]), dims=(1,)), dims=(1,))

# MVN STATIC
sta_swp = sta_data[:swp_ind] |> Array
sta_swp .+= 1
sta_time = sta_data[:epoch] |> Array
sta_ind = ThreadsX.findall(ts .<= sta_data[:epoch] .<= te)
sta_julian=datetime2julian.(DateTime.(sta_time))
sta_eflux = sta_data[:eflux] |> Array
sta_mass = sta_data[:mass_arr] |> Array
sta_energy = sta_data[:energy] |> Array
sta_theta = sta_data[:theta] |> Array
sta_phi = sta_data[:phi] |> Array
sta_denergy = sta_data[:denergy] |> Array
sta_dtheta = sta_data[:dtheta] |> Array
sta_dphi = sta_data[:dphi] |> Array
sta_domega = sta_data[:domega] |> Array
sta_velocity = sqrt.(sta_energy.*(2.0/ustrip(u"eV", m_p*c_0^2))./sta_mass).*ustrip(u"km/s", c_0)
sta_pos_sc_mso = sta_data[:pos_sc_mso] |> Array
sta_pos_sc_mso /= ustrip(u"km", R_Mars)
sta_quat_mso = sta_data[:quat_mso] |> Array
sta_quat_sc = sta_data[:quat_sc] |> Array
sta_mso_trans = [QuatRotation(quat) for quat in eachcol(sta_quat_mso')]
sta_swp_ind = sta_swp[sta_ind[1]]
sta_eflux_mean = dropdims(mean(sta_eflux[sta_ind, :, :, :], dims = (1, ) ), dims=(1, ))

sta_ind_mean = round(Int64, mean(sta_ind))
sta_sta2mso_mean = QuatRotation(sta_quat_mso[sta_ind_mean, :])
B_mean_sta = sta_sta2mso_mean' * mag_B_mean_mso
para_sta = B_mean_sta ./ norm(B_mean_sta)

3-element StaticArraysCore.SVector{3, Float32} with indices SOneTo(3):
 -0.9369204
 -0.31724733
 -0.14674586

In [3]:
swi, swj =2, 12
ind = argmax(sta_eflux_mean[1, (swj-1)*4+swi, :] ./ sta_energy[1, 1, :, sta_swp_ind, 1].^2)
sta_vswrough = CartesianFromSpherical()(CoordinateTransformations.Spherical(
    sta_velocity[1, (swj-1)*4+swi, ind, sta_swp_ind, 1], 
    deg2rad(sta_phi[1, (swj-1)*4+swi, ind, sta_swp_ind, 1]), 
    deg2rad(sta_theta[1, (swj-1)*4+swi, ind, sta_swp_ind, 1])))

ex = sta_vswrough / norm(sta_vswrough)
ey = B_mean_sta / norm(B_mean_sta)
ez = cross(ex, ey)

3-element StaticArraysCore.SVector{3, Float64} with indices SOneTo(3):
 -0.08074882255107417
 -0.186257712034746
  0.9182199824947902

In [11]:
sta_velocity[1, :, :, sta_swp_ind]
# sta_phi

64×32 Matrix{Float64}:
 2248.61  1876.29  1574.05  1320.47  …  14.8987  12.4409  10.3885  8.6747
 2248.61  1876.29  1574.05  1320.47     14.8987  12.4409  10.3885  8.6747
 2248.61  1876.29  1574.05  1320.47     14.8987  12.4409  10.3885  8.6747
 2248.61  1876.29  1574.05  1320.47     14.8987  12.4409  10.3885  8.6747
 2248.61  1876.29  1574.05  1320.47     14.8987  12.4409  10.3885  8.6747
 2248.61  1876.29  1574.05  1320.47  …  14.8987  12.4409  10.3885  8.6747
 2248.61  1876.29  1574.05  1320.47     14.8987  12.4409  10.3885  8.6747
 2248.61  1876.29  1574.05  1320.47     14.8987  12.4409  10.3885  8.6747
 2248.61  1876.29  1574.05  1320.47     14.8987  12.4409  10.3885  8.6747
 2248.61  1876.29  1574.05  1320.47     14.8987  12.4409  10.3885  8.6747
 2248.61  1876.29  1574.05  1320.47  …  14.8987  12.4409  10.3885  8.6747
 2248.61  1876.29  1574.05  1320.47     14.8987  12.4409  10.3885  8.6747
 2248.61  1876.29  1574.05  1320.47     14.8987  12.4409  10.3885  8.6747
    ⋮          

In [14]:
sta_eflux[sta_ind, 1, :, :]

45×64×32 Array{Float32, 3}:
[:, :, 1] =
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0